# Dyck: 4 Models + Probes

This notebook is the main task-specific entry point for the Dyck experiments in `experiment_pipeline_plan.md`.

It trains four model families on one Dyck setting, extracts hidden states, runs sufficient-statistic probes, and writes the run summaries to `results/notebooks/dyck/<task_name>/`.

Expected model set:
- `rnn`
- `lstm`
- `transformer`
- `mamba` using the official `mamba-ssm` package


## 0. Path Setup

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT


## 1. Imports

In [ ]:
import pandas as pd
import torch

from hse.experiments.dyck import (
    DEFAULT_DYCK_TASKS,
    official_mamba_status,
    resolve_dyck_model_specs,
    run_dyck_suite,
)


## 2. Experiment Settings

Default values below are a smoke configuration so `Run All` finishes quickly.

For the real experiment in the plan, switch to something closer to:

```python
TRAINING_STEPS = 10_000
BATCH_SIZE = 128
EXTRACT_EXAMPLES = 50_000
SEEDS = [0, 1, 2]
```


In [ ]:
TASK_NAME = "dyck_no_noise"  # or "dyck_50_noise"
SEEDS = [0]
TRAINING_STEPS = 200
BATCH_SIZE = 32
EXTRACT_EXAMPLES = 512
EXTRACT_BATCH_SIZE = 256
LEARNING_RATE = 3e-4
EVAL_EVERY = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Keep this True if you want the notebook to enforce the official 4-model comparison.
REQUIRE_OFFICIAL_MAMBA = True
FALLBACK_TO_MAMBA_LIKE = False

RESULTS_ROOT = ROOT / "results" / "notebooks" / "dyck" / TASK_NAME
RESULTS_ROOT


## 3. Preflight Checks

This cell fails early if official `mamba-ssm` is missing while `REQUIRE_OFFICIAL_MAMBA = True`.


In [ ]:
assert TASK_NAME in DEFAULT_DYCK_TASKS, f"Unknown task: {TASK_NAME}"

mamba_status = official_mamba_status()
mamba_status


In [ ]:
model_specs = resolve_dyck_model_specs(
    require_official_mamba=REQUIRE_OFFICIAL_MAMBA,
    fallback_to_mamba_like=FALLBACK_TO_MAMBA_LIKE,
)
model_specs


## 4. Run Dyck

This cell trains all selected models, extracts hidden states, and runs the probes.


In [ ]:
runs_df, probe_df = run_dyck_suite(
    task_name=TASK_NAME,
    seeds=SEEDS,
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    extract_examples=EXTRACT_EXAMPLES,
    extract_batch_size=EXTRACT_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY,
    device=DEVICE,
    results_root=RESULTS_ROOT,
    require_official_mamba=REQUIRE_OFFICIAL_MAMBA,
    fallback_to_mamba_like=FALLBACK_TO_MAMBA_LIKE,
)


## 5. Inspect Results

In [ ]:
runs_df


In [ ]:
probe_df


In [ ]:
sorted(str(p.relative_to(ROOT)) for p in RESULTS_ROOT.rglob("summary.json"))
